# SelfSNE quick demo

Minimal Self-Supervised Noise Embedding training on synthetic data.

In [ ]:
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader, TensorDataset
from selfsne import SelfSNE
from selfsne.losses import LikelihoodRatioEstimator

In [ ]:
pl.seed_everything(0)
t = torch.linspace(0, 4*torch.pi, 256)
x = torch.stack([t.sin(), t.cos(), (2*t).sin(), (2*t).cos()], dim=1)
ds = TensorDataset(x[:-1], x[1:])
loader = DataLoader(ds, batch_size=64)

In [ ]:
model = SelfSNE(
    loss=LikelihoodRatioEstimator(divergence="bce"),
    target_encoder=torch.nn.Sequential(
        torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 2)
    )
)

trainer = pl.Trainer(max_epochs=3, accelerator="cpu", devices=1, logger=False, enable_checkpointing=False, enable_progress_bar=False)
trainer.fit(model, loader)
emb = trainer.predict(model, loader)
embeddings = torch.cat([b["target_embedding"] for b in emb])
print("Embeddings shape:", embeddings.shape)

In [ ]:
# Test SelfSNE's information preservation: linear probe from embeddings -> t
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
X = embeddings
y = t[1:]
probe = nn.Linear(2, 1)
opt = torch.optim.Adam(probe.parameters(), lr=1e-2)
ds = TensorDataset(X, y.unsqueeze(1))
dl = DataLoader(ds, batch_size=64, shuffle=True)
for _ in range(200):
    for xb, yb in dl:
        opt.zero_grad()
        pred = probe(xb)
        loss = nn.MSELoss()(pred, yb)
        loss.backward()
        opt.step()
with torch.no_grad():
    pred = probe(X)
    ss_res = ((y.unsqueeze(1) - pred)**2).sum()
    ss_tot = ((y.unsqueeze(1) - y.mean())**2).sum()
    r2 = 1 - ss_res/ss_tot
print("Linear probe R^2:", r2.item())